# EX05 — Results Analysis Notebook

**Authors:** Itay Malich & Diana Koroblov  
**Course:** On-Premises LLM Deployment (L08)

This notebook is the project's central research artifact: it loads the raw
`results/*.json` produced by the experiments and reproduces the key tables,
comparisons, and derivations behind the README. Run top-to-bottom; it is
fully offline and depends only on the repo's declared packages
(`numpy`, `matplotlib`).

Regenerate the inputs with `uv run python experiments/run_baseline.py`,
`run_airllm.py`, `run_ollama.py`, and `run_economics.py`.

## 1. Load raw results

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'

def latest_by_scenario():
    """Return the newest KPI result per scenario (skips economics)."""
    out = {}
    for p in sorted(RESULTS.glob('*.json')):
        if 'economics' in p.name:
            continue
        d = json.loads(p.read_text(encoding='utf-8'))
        out[d.get('scenario', p.stem)] = d
    return out

results = latest_by_scenario()
print(f'Loaded {len(results)} scenarios:')
for s in results:
    print(' -', s)

## 2. Master KPI table

All scenarios across all three stages (Baseline, AirLLM, Ollama).

In [ ]:
COLS = ['ttft_seconds', 'tpot_seconds', 'throughput_tokens_per_sec',
        'peak_ram_gb', 'estimated_power_wh']
HEAD = ['Scenario', 'TTFT(s)', 'TPOT(s)', 'tok/s', 'RAM(GB)', 'Wh', 'error']

def row(name, d):
    vals = [f"{d.get(c, 0):.3f}" if d.get(c) else '-' for c in COLS]
    return [name] + vals + [(d.get('error') or 'ok')[:24]]

rows = [row(s, d) for s, d in results.items()]
widths = [max(len(str(r[i])) for r in [HEAD] + rows) for i in range(len(HEAD))]
fmt = '  '.join('{:<' + str(w) + '}' for w in widths)
print(fmt.format(*HEAD))
print(fmt.format(*['-' * w for w in widths]))
for r in rows:
    print(fmt.format(*r))

## 3. Stage 1 vs Stage 2 — the AirLLM trade-off

AirLLM's headline result: it converts an un-loadable model into a runnable
(but slow) one by streaming layers from disk.

In [ ]:
base = results.get('baseline_fp16', {})
air = results.get('airllm_fp16', {})
if base and air:
    ram_factor = base['peak_ram_gb'] / air['peak_ram_gb']
    lat_factor = air['tpot_seconds'] / 0.62  # baseline est. decode s/tok
    print(f"Baseline peak RAM : {base['peak_ram_gb']:.2f} GB")
    print(f"AirLLM   peak RAM : {air['peak_ram_gb']:.2f} GB")
    print(f"-> RAM reduction  : {ram_factor:.1f}x")
    print(f"-> Latency cost   : ~{lat_factor:.0f}x slower decode")

## 4. Stage 3 — real CPU quantization gradient (Ollama / GGUF)

The positive quantization result: as precision drops, RAM falls and
throughput rises. (AirLLM's Q4/Q8 could not run — bitsandbytes needs CUDA.)

In [ ]:
import matplotlib.pyplot as plt

order = ['ollama_fp16', 'ollama_q8', 'ollama_q4', 'ollama_q2']
labels = ['FP16', 'Q8', 'Q4', 'Q2']
present = [(l, s) for l, s in zip(labels, order) if s in results]
labels = [l for l, s in present]
ram = [results[s]['peak_ram_gb'] for l, s in present]
tput = [results[s]['throughput_tokens_per_sec'] for l, s in present]

for l, r, t in zip(labels, ram, tput):
    print(f'{l:5s} RAM={r:6.2f} GB   throughput={t:6.2f} tok/s')

if present:
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
    a.bar(labels, ram, color='#4c72b0'); a.set_title('Peak RAM (GB)')
    b.bar(labels, tput, color='#55a868'); b.set_title('Throughput (tok/s)')
    for ax in (a, b):
        ax.set_xlabel('Quantization (precision decreasing ->)')
    plt.tight_layout(); plt.show()

## 5. Roofline derivation

Decode is a GEMV: each 2-byte weight is read once for 2 FLOP, so the
arithmetic intensity is

$$ I_{decode} = \frac{2\ \text{FLOP}}{2\ \text{byte}} = 1\ \text{FLOP/byte}. $$

Attainable performance under a memory roof of bandwidth $BW$ is
$P(I) = \min(P_{peak},\, I \cdot BW)$, and the ridge point (where memory
and compute roofs meet) is $I^* = P_{peak} / BW$. With $I_{decode}=1$ far
below either ridge, decode is **memory-bound** on this hardware.

In [ ]:
PEAK_TFLOPS = 2560.0  # GFLOP/s, 8c x 5GHz x 64 FLOP/cycle (AVX-512)
DDR5, NVME = 50.0, 7.0  # GB/s
PARAMS = 8.03e9
FLOP_PER_TOK = 2 * PARAMS

for name, bw in [('DDR5', DDR5), ('NVMe', NVME)]:
    print(f'{name}: ridge I* = {PEAK_TFLOPS / bw:.0f} FLOP/byte')

for name, tpot in [('Baseline (DDR5)', 0.62), ('AirLLM (NVMe)', air.get('tpot_seconds', 15.09))]:
    perf = FLOP_PER_TOK / tpot / 1e9
    print(f'{name:18s} decode = {perf:6.2f} GFLOP/s  (at I=1)')

## 6. Economics — break-even

In [ ]:
econ_files = sorted(RESULTS.glob('economics_*.json'))
if econ_files:
    e = json.loads(econ_files[-1].read_text(encoding='utf-8'))
    print(f"API cost/req (no cache): ILS {e['api_cost_per_request_ils']:.6f}")
    print(f"Cloud GPU cost/req     : ILS {e['cloud_gpu_cost_per_request_ils']:.6f}")
    print(f"Break-even (no cache)  : {e['breakeven_requests_no_cache']:,} req/mo")
    print(f"Break-even (cached)    : {e['breakeven_requests_cached']:,} req/mo")

## 7. Conclusions

1. **Bottleneck = memory, not compute** (capacity + bandwidth; VRAM unused).
2. **AirLLM** trades ~6.7x RAM for ~24x latency — fits the un-fittable.
3. **Quantization works on CPU via GGUF**, not CUDA-only bitsandbytes; RAM
   falls 4.4x and throughput rises 4.5x FP16->Q2; **Q4 is the sweet spot**.
4. **Economically**, the API wins below ~525k req/mo for this workload.

### References

- Williams, Waterman, Patterson. *Roofline: An Insightful Visual Performance
  Model for Multicore Architectures.* CACM, 2009.
- AirLLM — https://github.com/lyogavin/airllm
- llama.cpp / GGUF — https://github.com/ggml-org/llama.cpp